# Step 1 - Preprocessing Walkthrough

This notebook is the teaching version of `src/01_preprocessing.py`.

Audience:
- Readers who want to understand how the project turns raw trade data into a model-ready annual dataset.

Prerequisites:
- Basic comfort with pandas tables
- A rough idea of missing values, rolling averages, and lagged features

Learning goals:
- Understand the role of the annual, country, and raw input files
- Audit key columns for missing values
- Engineer the extra features used later in the pipeline
- Save the enriched dataset and preprocessing report expected by the repo


## Outline

1. Set up imports and project paths
2. Load the three input datasets
3. Inspect key columns and run a missing-value audit
4. Engineer the derived features used later in modeling
5. Compare the pre-2018 and post-2018 periods
6. Review the policy dummy variables
7. Save the enriched CSV and text report
8. Try a short feature-engineering exercise


In [1]:
from pathlib import Path
import os
import sys
import warnings
filterwarnings = True
if filterwarnings:
    import warnings
    warnings.filterwarnings("ignore")

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(path for path in candidates if (path / "config.py").exists())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from config import ANNUAL_FILE, BREAK_YEAR, COUNTRY_FILE, RAW_FILE, REPORT_DIR

warnings.filterwarnings(
    "ignore",
    message="FigureCanvasAgg is non-interactive.*",
    category=UserWarning,
)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT


WindowsPath('C:/Users/Pegasus/Desktop/SemiTrack2')

## Load the input data

This step uses three related datasets. The annual file is the main modeling table, the country file helps us understand supplier patterns, and the raw transaction file is the low-level source that feeds the processed outputs.


In [2]:
ann = pd.read_csv(ANNUAL_FILE)
ann["year"] = ann["year"].astype(int)
ann = ann.sort_values("year").reset_index(drop=True)

cty = pd.read_csv(COUNTRY_FILE)
cty["year"] = cty["year"].astype(int)

raw = pd.read_csv(RAW_FILE, encoding="utf-8-sig")
raw["year"] = raw["year"].astype(int)

pd.DataFrame(
    {
        "dataset": ["annual integrated", "country breakdown", "raw transactions"],
        "rows": [ann.shape[0], cty.shape[0], raw.shape[0]],
        "columns": [ann.shape[1], cty.shape[1], raw.shape[1]],
        "purpose": [
            "Main yearly modeling dataset",
            "Supplier-country detail for market structure analysis",
            "Original transaction-level source table",
        ],
    }
)


,dataset,rows,columns,purpose
0,annual integrated,30,35,Main yearly modeling dataset
1,country breakdown,2253,11,Supplier-country detail for market structure a...
2,raw transactions,6261,9,Original transaction-level source table


## Preview the annual table

Before we engineer anything, it helps to look at the core yearly fields the rest of the pipeline depends on. This also makes the later derived columns easier to interpret.


In [3]:
ann[
    [
        "year",
        "import_value_usd_billions",
        "real_value_2015usd_billions",
        "yoy_real_growth_pct",
        "china_share_pct",
        "supplier_hhi",
    ]
].head(8)


,year,import_value_usd_billions,real_value_2015usd_billions,yoy_real_growth_pct,china_share_pct,supplier_hhi
0,1995,0.2374,0.8695,NaN,0.7800,0.0511
1,1996,0.1541,0.5107,-41.2600,0.7200,0.0840
2,1997,0.2296,0.7070,38.4300,1.6100,0.0792
3,1998,0.2462,0.6799,-3.8400,3.0400,0.0622
4,1999,0.3221,0.8458,24.4000,2.6100,0.0617
5,2000,0.3358,0.8422,-0.4200,2.9700,0.0583
6,2001,0.3394,0.8053,-4.3700,2.9400,0.0582
7,2002,0.3582,0.8026,-0.3400,5.2700,0.0459


## Audit missing values in the key modeling columns

A preprocessing script should answer a simple question first: are the important columns complete enough to trust? Here we focus on the fields used directly in later analysis and forecasting steps.


In [4]:
key_cols = [
    "import_value_usd_billions",
    "real_value_2015usd_billions",
    "hs8542_import_usd",
    "hs3818_import_usd",
    "china_share_pct",
    "supplier_hhi",
    "yoy_nominal_growth_pct",
    "yoy_real_growth_pct",
]

audit_rows = []
for column in key_cols:
    missing_count = int(ann[column].isna().sum())
    audit_rows.append(
        {
            "column": column,
            "missing_values": missing_count,
            "status": "OK" if missing_count <= 1 else f"WARN: {missing_count} missing",
        }
    )

audit_df = pd.DataFrame(audit_rows)
audit_df


,column,missing_values,status
0,import_value_usd_billions,0,OK
1,real_value_2015usd_billions,0,OK
2,hs8542_import_usd,0,OK
3,hs3818_import_usd,0,OK
4,china_share_pct,0,OK
5,supplier_hhi,0,OK
6,yoy_nominal_growth_pct,1,OK
7,yoy_real_growth_pct,1,OK


## Engineer the derived features

These extra columns translate the raw annual series into forms that are more useful for modeling. Some capture growth dynamics, some add memory through lags, and some describe market concentration in a more flexible way.


In [5]:
ann_enriched = ann.copy()
ann_enriched["log_real_import"] = np.log(ann_enriched["real_value_2015usd_billions"])
ann_enriched["log_diff"] = ann_enriched["log_real_import"].diff()
ann_enriched["first_diff_real"] = ann_enriched["real_value_2015usd_billions"].diff()
ann_enriched["rolling3_real"] = ann_enriched["real_value_2015usd_billions"].rolling(3, min_periods=1).mean()
ann_enriched["lag1_real"] = ann_enriched["real_value_2015usd_billions"].shift(1)
ann_enriched["lag2_real"] = ann_enriched["real_value_2015usd_billions"].shift(2)
ann_enriched["china_share_squared"] = ann_enriched["china_share_pct"] ** 2
ann_enriched["hhi_diff"] = ann_enriched["supplier_hhi"].diff()

features_added = [
    "log_real_import",
    "log_diff",
    "first_diff_real",
    "rolling3_real",
    "lag1_real",
    "lag2_real",
    "china_share_squared",
    "hhi_diff",
]

ann_enriched[["year", *features_added]].head(8)


,year,log_real_import,log_diff,first_diff_real,rolling3_real,lag1_real,lag2_real,china_share_squared,hhi_diff
0,1995,-0.1398,NaN,NaN,0.8695,NaN,NaN,0.6084,NaN
1,1996,-0.6720,-0.5321,-0.3588,0.6901,0.8695,NaN,0.5184,0.0329
2,1997,-0.3467,0.3252,0.1963,0.6957,0.5107,0.8695,2.5921,-0.0048
3,1998,-0.3858,-0.0391,-0.0271,0.6325,0.7070,0.5107,9.2416,-0.0170
4,1999,-0.1675,0.2183,0.1659,0.7442,0.6799,0.7070,6.8121,-0.0004
5,2000,-0.1717,-0.0043,-0.0036,0.7893,0.8458,0.6799,8.8209,-0.0035
6,2001,-0.2165,-0.0448,-0.0369,0.8311,0.8422,0.8458,8.6436,-0.0001
7,2002,-0.2199,-0.0034,-0.0027,0.8167,0.8053,0.8422,27.7729,-0.0123


## Read the engineered columns like an analyst

A feature is easier to trust when you can explain why it exists. The table below turns the raw feature list into plain-language meaning.


In [6]:
pd.DataFrame(
    {
        "feature": features_added,
        "what_it_means": [
            "Log of real imports to stabilize scale",
            "Year-to-year change in the logged series",
            "Absolute change in real imports",
            "Three-year smoothed real import level",
            "Previous year's real import value",
            "Two-years-back real import value",
            "Nonlinear version of China dependency",
            "Year-to-year change in supplier concentration",
        ],
        "why_it_matters": [
            "Useful for time-series modeling",
            "Connects directly to stationarity work",
            "Highlights annual jumps",
            "Smooths short-term noise",
            "Adds simple historical memory",
            "Adds slightly longer memory",
            "Lets later models capture curvature",
            "Tracks diversification or concentration shifts",
        ],
    }
)


,feature,what_it_means,why_it_matters
0,log_real_import,Log of real imports to stabilize scale,Useful for time-series modeling
1,log_diff,Year-to-year change in the logged series,Connects directly to stationarity work
2,first_diff_real,Absolute change in real imports,Highlights annual jumps
3,rolling3_real,Three-year smoothed real import level,Smooths short-term noise
4,lag1_real,Previous year's real import value,Adds simple historical memory
5,lag2_real,Two-years-back real import value,Adds slightly longer memory
6,china_share_squared,Nonlinear version of China dependency,Lets later models capture curvature
7,hhi_diff,Year-to-year change in supplier concentration,Tracks diversification or concentration shifts


## Compare the periods before and after 2018

The project treats 2018 as an important structural turning point. A simple way to build intuition is to compare average growth before and after that year, then look at the latest dependency snapshot.


In [7]:
pre = ann_enriched[ann_enriched["year"] < BREAK_YEAR]
post = ann_enriched[ann_enriched["year"] >= BREAK_YEAR]

summary_df = pd.DataFrame(
    {
        "metric": [
            f"Pre-{BREAK_YEAR} mean nominal growth (%)",
            f"Post-{BREAK_YEAR} mean nominal growth (%)",
            f"Pre-{BREAK_YEAR} mean real growth (%)",
            f"Post-{BREAK_YEAR} mean real growth (%)",
            "2024 China share (%)",
            "2024 Supplier HHI",
            "1995-2024 nominal growth multiple",
        ],
        "value": [
            pre["yoy_nominal_growth_pct"].mean(),
            post["yoy_nominal_growth_pct"].mean(),
            pre["yoy_real_growth_pct"].mean(),
            post["yoy_real_growth_pct"].mean(),
            ann_enriched.iloc[-1]["china_share_pct"],
            ann_enriched.iloc[-1]["supplier_hhi"],
            ann_enriched.iloc[-1]["import_value_usd_billions"] / ann_enriched.iloc[0]["import_value_usd_billions"],
        ],
    }
)
summary_df


,metric,value
0,Pre-2018 mean nominal growth (%),16.9832
1,Post-2018 mean nominal growth (%),42.3986
2,Pre-2018 mean real growth (%),9.9968
3,Post-2018 mean real growth (%),34.8786
4,2024 China share (%),45.8300
5,2024 Supplier HHI,0.2909
6,1995-2024 nominal growth multiple,97.0927


## Review the policy dummy variables

These dummy columns mark important events or regimes that later models may want to represent explicitly. Listing the active years makes the policy timeline much easier to verify.


In [8]:
dummy_cols = [column for column in ann_enriched.columns if column.startswith("dummy_")]
dummy_summary = []
for dummy in dummy_cols:
    dummy_summary.append(
        {
            "dummy": dummy,
            "active_years": ann_enriched.loc[ann_enriched[dummy] == 1, "year"].tolist(),
        }
    )

pd.DataFrame(dummy_summary)


,dummy,active_years
0,dummy_post_2018_inflection,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
1,dummy_covid_shock_2020,[2020]
2,dummy_global_chip_shortage_2021,[2021]
3,dummy_pli_scheme_launch,"[2021, 2022, 2023, 2024]"
4,dummy_micron_mou_2023,"[2023, 2024]"
5,dummy_tata_fab_groundbreaking,"[2023, 2024]"
6,dummy_india_semicon_mission,"[2021, 2022, 2023, 2024]"


## Save the enriched annual dataset

The pipeline expects a saved enriched CSV so later steps can reuse these engineered columns without recomputing them every time.


In [9]:
enriched_path = ANNUAL_FILE.replace(".csv", "_enriched.csv")
ann_enriched.to_csv(enriched_path, index=False)
enriched_path


'C:\\Users\\Pegasus\\Desktop\\SemiTrack2\\data\\processed\\india_semiconductor_integrated_annual_enriched.csv'

## Save the preprocessing report

The text report is the script-friendly summary of everything we just walked through. It records dataset sizes, missing-value checks, summary statistics, and the policy dummy timeline.


In [10]:
report_path = Path(REPORT_DIR) / "preprocessing_report.txt"
report_lines = [
    "SEMITRACK INDIA - Preprocessing Report",
    "=" * 65,
    "",
    f"Annual rows:        {ann_enriched.shape[0]}",
    f"Country rows:       {cty.shape[0]}",
    f"Raw rows:           {raw.shape[0]}",
    f"Years:              {ann_enriched['year'].min()} - {ann_enriched['year'].max()}",
    f"Features added:     {', '.join(features_added)}",
    "",
    "Missing value audit:",
]
for _, row in audit_df.iterrows():
    report_lines.append(f"  {row['column']}: {row['status']}")
report_lines.extend(
    [
        "",
        f"Pre-{BREAK_YEAR} mean real growth:   {pre['yoy_real_growth_pct'].mean():.2f}%",
        f"Post-{BREAK_YEAR} mean real growth:  {post['yoy_real_growth_pct'].mean():.2f}%",
        f"2024 import bill:   ${ann_enriched.iloc[-1]['import_value_usd_billions']:.2f}B",
        f"2024 China share:   {ann_enriched.iloc[-1]['china_share_pct']:.2f}%",
        f"2024 HHI:           {ann_enriched.iloc[-1]['supplier_hhi']:.4f}",
        "1995-2024 growth (nominal):  "
        f"{ann_enriched.iloc[-1]['import_value_usd_billions'] / ann_enriched.iloc[0]['import_value_usd_billions']:.1f}x",
        "",
        "Policy dummy variables:",
    ]
)
for row in dummy_summary:
    report_lines.append(f"  {row['dummy']}: {row['active_years']}")
report_path.write_text("\n".join(report_lines), encoding="utf-8")
str(report_path)


'C:\\Users\\Pegasus\\Desktop\\SemiTrack2\\outputs\\reports\\preprocessing_report.txt'

## Exercise

Add one more rolling feature with a different window, such as a 5-year moving average.

Questions to think about:
- Does the longer window look smoother than `rolling3_real`?
- What signal might you lose when you smooth too aggressively?


In [11]:
# ann_enriched["rolling5_real"] = ann_enriched["real_value_2015usd_billions"].rolling(5, min_periods=1).mean()
# ann_enriched[["year", "real_value_2015usd_billions", "rolling3_real", "rolling5_real"]].tail(10)


## Common pitfall

Feature engineering is only helpful if the meaning of each column stays clear. A notebook like this is valuable because it keeps the "what" and the "why" of each engineered feature visible together.
